In [ ]:
import os
import pandas as pd
import geopandas as gpd
from config.config import DATA_RAW, CRS_BNG, DATA_INTERIM, LAD_CODE_LONDON

Load geopackage boundary file

In [ ]:
gdf = gpd.read_file(filename=DATA_RAW / "boundaries" / "Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BFC_V10_-672099234420024429.gpkg")


In [ ]:
gdf

Check CRS and reproject if necessary

In [ ]:
if gdf.crs.to_epsg() != CRS_BNG:
    gdf = gdf.to_crs(CRS_BNG)

Load lookup and bring LAD codes across to gdf

In [ ]:
# Load lookup
lookup = pd.read_csv(filepath_or_buffer=DATA_RAW / "boundaries" / "LSOA_(2011)_to_LSOA_(2021)_to_Local_Authority_District_(2022)_Exact_Fit_Lookup_for_EW_(V3).csv")

gdf = gdf.merge(lookup[["LSOA21CD", "LAD22CD", "LAD22NM"]],
                on="LSOA21CD",
                how="left")

Filter to London only

In [ ]:
london_gdf = gdf[gdf["LAD22CD"].str.startswith(LAD_CODE_LONDON)].copy()

print(f"London LSOAs: {len(london_gdf)}")
print(f"Null LAD codes: {london_gdf['LAD22CD'].isna().sum()}")

Save file

In [ ]:
# Create interim directory if it doesn't already exist
os.makedirs(DATA_INTERIM / "boundaries", exist_ok=True)

london_gdf.to_file(
    DATA_INTERIM / "boundaries" / "london_lsoa_2021.gpkg",
    layer="london_lsoa_2021",
    driver="GPKG"
)

print("Saved successfully")